[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camenduru/gaussian-splatting-colab/blob/main/gaussian_splatting_viewer_colab.ipynb)

## Gaussian Splatting Viewer

Run this after `gaussian_splatting_colab.ipynb` has finished training.

It will:
1. Clone the WebGL splat viewer
2. Copy your trained `point_cloud.ply` into the viewer's directory
3. Open a public Cloudflare tunnel and print a direct URL to view your scene

In [ ]:
import os, shutil, glob

# Path where train.py writes its output (matches gaussian_splatting_colab.ipynb Step 5)
MODEL_PATH = '/content/my_scene/output'

# Find the latest point_cloud.ply — prefers iteration_30000, falls back to whatever exists
ply_candidates = sorted(glob.glob(f'{MODEL_PATH}/point_cloud/iteration_*/point_cloud.ply'), reverse=True)

if not ply_candidates:
    raise FileNotFoundError(
        f'No point_cloud.ply found under {MODEL_PATH}.\n'
        'Make sure gaussian_splatting_colab.ipynb Step 5 (training) completed successfully.'
    )

PLY_PATH = ply_candidates[0]
print(f'Using: {PLY_PATH}')

In [ ]:
!git clone https://github.com/camenduru/splat /content/splat
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared-linux-amd64.deb

In [ ]:
import atexit, requests, subprocess, time, re, os, shutil
from random import randint
from threading import Timer
from queue import Queue

# Copy the trained PLY into the splat viewer's static directory
VIEWER_PLY = '/content/splat/point_cloud.ply'
shutil.copy(PLY_PATH, VIEWER_PLY)
print(f'Copied PLY to viewer: {VIEWER_PLY}')

def cloudflared(port, metrics_port, output_queue):
    atexit.register(
        lambda p: p.terminate(),
        subprocess.Popen(
            ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'],
            stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
        )
    )
    attempts, tunnel_url = 0, None
    while attempts < 10 and not tunnel_url:
        attempts += 1
        time.sleep(3)
        try:
            tunnel_url = re.search(
                r'(?P<url>https?:\/\/[^\s]+\.trycloudflare\.com)',
                requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text
            ).group('url')
        except:
            pass
    if not tunnel_url:
        raise Exception("Can't connect to Cloudflare Edge")
    output_queue.put(tunnel_url)

output_queue = Queue()
metrics_port = randint(8100, 9000)
thread = Timer(2, cloudflared, args=(7860, metrics_port, output_queue))
thread.start()
thread.join()
tunnel_url = output_queue.get()
os.environ['webui_url'] = tunnel_url

# The splat viewer accepts ?url= to auto-load a file served from the same origin
direct_url = f'{tunnel_url}?url=point_cloud.ply'
print(f'\n Open your scene directly:\n  {direct_url}\n')
print(f'(Or open the base viewer and drag-drop point_cloud.ply manually: {tunnel_url})')

%cd /content/splat
!python -m http.server 7860